In [ ]:
import os, warnings
from pathlib import Path
from typing import List
import pandas as pd
from dotenv import load_dotenv
from ragas import evaluate, EvaluationDataset
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import (
    Faithfulness,
    FactualCorrectness,
    ResponseGroundedness, ContextRelevance, AnswerAccuracy
)
from ragas.llms import llm_factory
from openai import OpenAI
warnings.filterwarnings("ignore")
load_dotenv()

In [ ]:
TESTSET_PATH = Path("magisterka_eval/tags/testset_clean.csv")
QA_ENDPOINT = "http://localhost:8080/qa"
os.environ.get("OPENAI_API_KEY")
LLM_MODEL = "gpt-4o-mini"
evaluator_llm = llm_factory(LLM_MODEL, client=OpenAI())
print(f"LLM: {LLM_MODEL}")

# Optional - QA call (duplicates must be removed before QA)

In [ ]:
df = pd.read_csv(TESTSET_PATH, encoding="utf-8-sig")
RAW_SAMPLES = [{"question": q} for q in df["user_input"].dropna().str.strip() if q]

print(f"Loaded {len(RAW_SAMPLES)} questions from {TESTSET_PATH.name}")

In [ ]:
import httpx

async def call_qa(question: str, rag_mode: str, timeout: float = 600.0):
    payload = {"question": question}
    async with httpx.AsyncClient(timeout=timeout) as client:
        resp = await client.post(
            QA_ENDPOINT,
            json=payload,
            params={"rag_mode": rag_mode},
        )
        resp.raise_for_status()
        data = resp.json()

    answer = data.get("answer")
    reasoning = data.get("reasoning") or ""
    retrieved = data.get("source_context")
    return {"response": answer, "retrieved_contexts": retrieved, "raw": data, "reasoning": reasoning}


async def run_all_modes(samples, modes=("vanilla", "hybrid", "graph")):
    results = {mode: [] for mode in modes}
    total = len(samples) * len(modes)
    done = 0
    for s in samples:
        for mode in modes:
            try:
                out = await call_qa(s["question"], mode)
            except Exception as e:
                print(f"[{mode}] error for '{s['question'][:40]}': {e}")
                out = {"response": "", "retrieved_contexts": [], "raw": {}, "reasoning": ""}
            results[mode].append({
                **s,
                "response": out["response"],
                "retrieved_contexts": out["retrieved_contexts"],
                "reasoning": out["reasoning"],
                "mode": mode,
            })
            done += 1
            if done % 5 == 0:
                print(f"  [{done}/{total}] ...")
    return results

print("Calling the QA service for all samples x 3 modes...")
qa_results = await run_all_modes(RAW_SAMPLES)
print("Done!")
for mode, rows in qa_results.items():
    non_empty = sum(1 for r in rows if r["response"])
    with_reasoning = sum(1 for r in rows if r["reasoning"])
    print(f"  {mode:8}: {non_empty}/{len(rows)} answers")
    print(f"  {mode:8}: {non_empty}/{len(rows)} answers, reasoning: {with_reasoning}/{len(rows)}")

In [ ]:
from datetime import datetime
import csv

def save_qa_results_to_csv(qa_results, output_path: str):
    rows = []
    for mode, samples in qa_results.items():
        for r in samples:
            ctx = r.get("retrieved_contexts", "")
            if isinstance(ctx, list):
                ctx = "\n---\n".join(str(c) for c in ctx if c)

            rows.append({
                "mode": mode,
                "question": r.get("question", ""),
                "reasoning": r.get("reasoning", ""),
                "response": r.get("response", ""),
                "retrieved_contexts": ctx
            })

    df = pd.DataFrame(rows)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
        quoting=csv.QUOTE_ALL,
    )
    print(f"Saved {len(df)} rows -> {output_path}")

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
save_qa_results_to_csv(qa_results, output_path=f"magisterka_eval/tags/v5/qa_results_v5.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("magisterka_eval/tags/v5/qa_results_v5.csv")
df = df.rename(columns={"question": "user_input", "rag_mode": "mode"})
df = df.drop_duplicates(subset=["user_input", "mode"])

for col in ["faithfulness", "factual_correctness(mode=f1)", "response_groundedness", "context_relevance", "answer_accuracy"]:
    df[col] = None

target_columns = [
    "user_input", "retrieved_contexts", "reference_contexts",
    "response", "reasoning", "reference", "mode",
    "faithfulness", "factual_correctness(mode=f1)",
    "response_groundedness", "context_relevance", "answer_accuracy",
]
df = df.reindex(columns=target_columns)

output_path = "magisterka_eval/tags/v5/merged_final_v5.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} rows -> {output_path}")
print(f"Missing reference: {df['reference'].isna().sum()}")
print(f"Missing reference_contexts: {df['reference_contexts'].isna().sum()}")

In [ ]:
testset = pd.read_csv(TESTSET_PATH)
ref_cols = testset[["user_input", "reference", "reference_contexts"]].drop_duplicates("user_input")

df = df.drop(columns=["reference", "reference_contexts"], errors="ignore")
df = df.merge(ref_cols, on="user_input", how="left")

print(f"Missing reference:          {df['reference'].isna().sum()}")
print(f"Missing reference_contexts: {df['reference_contexts'].isna().sum()}")

df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} rows -> {output_path}")

In [ ]:
v4 = pd.read_csv("magisterka_eval/tags/v4/merged_final_v4.csv")
v5 = pd.read_csv("magisterka_eval/tags/v5/merged_final_v5.csv")

print("v4:", v4.columns.tolist())
print("v5:", v5.columns.tolist())

v5 = v5.reindex(columns=v4.columns)
v5.to_csv("magisterka_eval/tags/v5/merged_final_v5.csv",
          index=False, encoding="utf-8-sig")
print("Done. Columns:", v5.columns.tolist())

# 1 row - 1 call

In [ ]:
import math
import ast
from ragas import RunConfig
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

MISSING_COLS = ["faithfulness", "factual_correctness(mode=f1)"]

_chat_llm = ChatOpenAI(
    model=LLM_MODEL,
    max_tokens=16384
)
eval_llm_high = LangchainLLMWrapper(_chat_llm)

METRIC_COL_PAIRS = [
    (Faithfulness(llm=eval_llm_high),      "faithfulness"),
    (FactualCorrectness(llm=eval_llm_high), "factual_correctness(mode=f1)"),
]

df_full = pd.read_csv(TESTSET_PATH)
missing_mask = df_full[MISSING_COLS].isnull().any(axis=1)
df_missing = df_full[missing_mask].copy()
print(f"Missing rows: {len(df_missing)} / {len(df_full)}")

if df_missing.empty:
    print("Nothing missing - nothing to do.")
else:
    def safe_str(val, fallback="") -> str:
        if val is None: return fallback
        if isinstance(val, float) and math.isnan(val): return fallback
        return str(val).strip() or fallback

    def safe_ctx(val) -> List[str]:
        if val is None: return ["brak kontekstu"]
        if isinstance(val, float) and math.isnan(val): return ["brak kontekstu"]
        if isinstance(val, list): return val if val else ["brak kontekstu"]
        s = str(val).strip()
        if not s: return ["brak kontekstu"]
        if s.startswith("["):
            try:
                parsed = ast.literal_eval(s)
                return parsed if parsed else ["brak kontekstu"]
            except: pass
        return [s]

    def make_sample(row):
        return SingleTurnSample(
            user_input=safe_str(row.get("user_input") or row.get("question"), "brak pytania"),
            response=safe_str(row.get("response"), "brak odpowiedzi"),
            reference=safe_str(row.get("reference"), "brak referencji"),
            retrieved_contexts=safe_ctx(row.get("retrieved_contexts")),
            reference_contexts=safe_ctx(row.get("reference_contexts")),
        )

    def find_result_col(metric, res_row):
        if metric.name in res_row.index:
            return metric.name
        matches = [c for c in res_row.index if c.startswith(metric.name)]
        return matches[0] if matches else None

    run_cfg = RunConfig(max_retries=2, max_wait=360)

    for mode, grp in df_missing.groupby("mode"):
        pairs_to_run = [(m, col) for m, col in METRIC_COL_PAIRS if grp[col].isnull().any()]
        metrics = [m for m, _ in pairs_to_run]
        print(f"[{mode}] {len(grp)} rows | metrics: {[m.name for m in metrics]}")

        for orig_idx, row in grp.iterrows():
            if not any(pd.isnull(df_full.at[orig_idx, col]) for _, col in pairs_to_run):
                continue

            try:
                sample = make_sample(row)
            except Exception as e:
                print(f"Sample idx={orig_idx}: {e}")
                continue

            ds = EvaluationDataset(samples=[sample])
            try:
                result = evaluate(
                    dataset=ds,
                    metrics=metrics,
                    run_config=run_cfg,
                    raise_exceptions=False,
                )
                res_row = result.to_pandas().iloc[0]

                for metric, col in pairs_to_run:
                    if not pd.isnull(df_full.at[orig_idx, col]):
                        continue

                    result_col = find_result_col(metric, res_row)
                    if result_col is None:
                        print(f"idx={orig_idx}: no column for '{metric.name}' in {list(res_row.index)}")
                        continue

                    val = res_row[result_col]
                    if isinstance(val, float) and math.isnan(val):
                        print(f"idx={orig_idx} {metric.name}=NaN")
                        continue

                    df_full.at[orig_idx, col] = val
                    print(f"idx={orig_idx} {col}={val:.3f}")

            except Exception as e:
                print(f"idx={orig_idx} evaluation error: {e}")
                continue

        df_full.to_csv(TESTSET_PATH, index=False)
        print(f"[{mode}] intermediate results saved")

    still_missing = df_full[MISSING_COLS].isnull().any(axis=1).sum()
    print(f"Saved {TESTSET_PATH}")
    print(f"Still missing: {still_missing} rows" + (" (API errors?)" if still_missing else " ok"))

# Batch with fallback

In [ ]:
import time
import math
import ast
from ragas import RunConfig
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper

BATCH_CHUNK = 10
MERGED_CSV_PATH = r"magisterka_eval/tags/v5/merged_final_v5.csv"

MISSING_COLS = [
    "faithfulness",
    "factual_correctness(mode=f1)",
    "response_groundedness",
    "context_relevance",
    "answer_accuracy",
]


_chat_llm = ChatOpenAI(
    model=LLM_MODEL,
    max_tokens=16384
)
eval_llm_high = LangchainLLMWrapper(_chat_llm)

METRIC_COL_PAIRS = [
    (Faithfulness(llm=eval_llm_high), "faithfulness"),
    (FactualCorrectness(llm=eval_llm_high), "factual_correctness(mode=f1)"),
    (ResponseGroundedness(llm=eval_llm_high), "response_groundedness"),
    (ContextRelevance(llm=eval_llm_high), "context_relevance"),
    (AnswerAccuracy(llm=eval_llm_high), "answer_accuracy"),
]

df_full = pd.read_csv(MERGED_CSV_PATH)
missing_mask = df_full[MISSING_COLS].isnull().any(axis=1)
df_missing = df_full[missing_mask].copy()
print(f"Missing rows: {len(df_missing)} / {len(df_full)}")

if df_missing.empty:
    print("Nothing missing - nothing to do.")
else:
    def safe_str(val, fallback="") -> str:
        if val is None: return fallback
        if isinstance(val, float) and math.isnan(val): return fallback
        return str(val).strip() or fallback

    def safe_ctx(val) -> List[str]:
        if val is None: return ["brak kontekstu"]
        if isinstance(val, float) and math.isnan(val): return ["brak kontekstu"]
        if isinstance(val, list): return val if val else ["brak kontekstu"]
        s = str(val).strip()
        if not s: return ["brak kontekstu"]
        if s.startswith("["):
            try:
                parsed = ast.literal_eval(s)
                return parsed if parsed else ["brak kontekstu"]
            except: pass
        return [s]

    def make_sample(row):
        return SingleTurnSample(
            user_input=safe_str(row.get("user_input") or row.get("question"), "brak pytania"),
            response=safe_str(row.get("response"), "brak odpowiedzi"),
            reference=safe_str(row.get("reference"), "brak referencji"),
            retrieved_contexts=safe_ctx(row.get("retrieved_contexts")),
            reference_contexts=safe_ctx(row.get("reference_contexts")),
        )

    def find_result_col(metric, res_row):
        if metric.name in res_row.index:
            return metric.name
        matches = [c for c in res_row.index if c.startswith(metric.name)]
        return matches[0] if matches else None

    run_cfg = RunConfig(max_retries=10, max_wait=60, timeout=120)

    for mode, grp in df_missing.groupby("mode"):
        pairs_to_run = [(m, col) for m, col in METRIC_COL_PAIRS if grp[col].isnull().any()]
        metrics = [m for m, _ in pairs_to_run]

        to_eval = [(idx, row) for idx, row in grp.iterrows()
                   if any(pd.isnull(df_full.at[idx, col]) for _, col in pairs_to_run)]

        chunks = [to_eval[i:i+BATCH_CHUNK] for i in range(0, len(to_eval), BATCH_CHUNK)]
        print(f"[{mode}] {len(to_eval)} rows -> {len(chunks)} chunks of ~{BATCH_CHUNK}")

        for chunk_i, chunk in enumerate(chunks):
            print(f"[{mode}] chunk {chunk_i+1}/{len(chunks)}...")
            try:
                samples = [make_sample(row) for _, row in chunk]
                result = evaluate(
                    dataset=EvaluationDataset(samples=samples),
                    metrics=metrics,
                    run_config=run_cfg,
                    raise_exceptions=False,
                )
                df_result = result.to_pandas()
                for (orig_idx, _), (_, res_row) in zip(chunk, df_result.iterrows()):
                    for metric, col in pairs_to_run:
                        if not pd.isnull(df_full.at[orig_idx, col]): continue
                        result_col = find_result_col(metric, res_row)
                        if result_col is None: continue
                        val = res_row[result_col]
                        if not (isinstance(val, float) and math.isnan(val)):
                            df_full.at[orig_idx, col] = val

                df_full.to_csv(MERGED_CSV_PATH, index=False)
                print(f"[{mode}] chunk {chunk_i+1} OK, saved")

            except (Exception, TimeoutError) as e:
                print(f"[{mode}] chunk {chunk_i+1} failed ({e}) - fallback 1-by-1")
                time.sleep(10)
                for orig_idx, row in chunk:
                    time.sleep(1)
                    if not any(pd.isnull(df_full.at[orig_idx, col]) for _, col in pairs_to_run):
                        continue
                    try:
                        result = evaluate(
                            dataset=EvaluationDataset(samples=[make_sample(row)]),
                            metrics=metrics,
                            run_config=run_cfg,
                            raise_exceptions=False,
                        )
                        res_row = result.to_pandas().iloc[0]
                        for metric, col in pairs_to_run:
                            if not pd.isnull(df_full.at[orig_idx, col]): continue
                            result_col = find_result_col(metric, res_row)
                            if result_col is None: continue
                            val = res_row[result_col]
                            if not (isinstance(val, float) and math.isnan(val)):
                                df_full.at[orig_idx, col] = val
                                print(f"idx={orig_idx} {col}={val:.3f}")

                    except Exception as e2:
                        time.sleep(2)
                        print(f"idx={orig_idx}: {e2}")

        df_full.to_csv(MERGED_CSV_PATH, index=False)
        print(f"[{mode}] saved")

    still_missing = df_full[MISSING_COLS].isnull().any(axis=1).sum()
    print(f"Saved {MERGED_CSV_PATH}")
    print(f"Still missing: {still_missing} rows" + (" (API errors?)" if still_missing else " ok"))

In [ ]:
still_missing = df_full[MISSING_COLS].isnull().any(axis=1).sum()
print(f"Still missing: {still_missing} rows" + (" (API errors?)" if still_missing else " ok"))